# Google Colab — быстрый старт

Этот ноутбук **один раз за сессию** клонирует репозиторий (или переходит в уже смонтированный проект), ставит зависимости и проверяет загрузку данных.

**GPU в Colab:** текущие модели (`04_models.ipynb`) — **scikit-learn**; они **не используют GPU**. Включение *Runtime → Change runtime type → GPU* не ускорит LR/Random Forest. GPU имело бы смысл при XGBoost-GPU, cuML, PyTorch и т.п.

**Перед запуском:** в ячейке ниже задайте `REPO_URL` (HTTPS URL вашего форка или оригинала на GitHub). Если репозиторий приватный — загрузите zip в Colab и используйте блок «Проект уже в Drive / вручную».

## 1. Клонирование и установка (`/content`)

Запустите ячейку ниже. Она создаст `/content/water-quality-ee` и выполнит `pip install`.

In [ ]:
import os
import subprocess
import sys

WORKDIR = "/content"
REPO_DIR = "water-quality-ee"
# Замените на URL своего клона (ветка main/master подставится из репо):
REPO_URL = "https://github.com/YOUR_USER/water-quality-ee.git"

os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)
repo_path = os.path.join(WORKDIR, REPO_DIR)

if not os.path.isdir(os.path.join(repo_path, ".git")):
    if "YOUR_USER" in REPO_URL:
        raise RuntimeError("Укажите реальный REPO_URL (GitHub HTTPS) в этой ячейке.")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", repo_path, "pull", "--ff-only"], check=False)

os.chdir(repo_path)
print("Рабочая папка:", os.getcwd())

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
print("pip install готов.")

## 2. (Опционально) Проект на Google Drive

Если вы скопировали репозиторий в Drive или клонируете туда вручную:
1. Раскомментируйте и выполните монтирование.
2. Укажите **реальный** путь `PROJECT_ON_DRIVE`.
3. Повторите `pip install` из этой папки (ячейка ниже).

Так кэш `data/raw/` можно сохранять между сессиями (папка `data/` в репозитории).

In [ ]:
# Раскомментируйте при использовании Drive:
# from google.colab import drive
# drive.mount("/content/drive")

# Пример: PROJECT_ON_DRIVE = "/content/drive/MyDrive/water-quality-ee"
PROJECT_ON_DRIVE = None  # или строка с путём

import os
import subprocess
import sys

if PROJECT_ON_DRIVE and os.path.isdir(PROJECT_ON_DRIVE):
    os.chdir(PROJECT_ON_DRIVE)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
    print("OK:", os.getcwd())
else:
    print("Drive-блок пропущен (PROJECT_ON_DRIVE не задан). Используйте шаг 1.")

## 3. Проверка: импорт и `load_domain`

При первом запросе данные скачаются с vtiav.sm.ee (нужен интернет).

In [ ]:
import importlib

import data_loader
importlib.reload(data_loader)

df = data_loader.load_domain("supluskoha", use_cache=True)
print("Проб:", len(df), "| колонки:", list(df.columns)[:6], "...")
print("compliant:", df["compliant"].value_counts(dropna=False).head().to_dict())

## 4. Дальнейшие ноутбуки (01→05)

1. В Colab слева откройте **файловый браузер** (иконка папки).
2. Перейдите в `water-quality-ee` → `notebooks`.
3. Откройте `01_eda_supluskoha.ipynb` (двойной клик / Open) и выполняйте по порядку.

Первая ячейка в `01`…`05` сама подхватит путь, если **текущая рабочая директория** — корень репозитория. После шага 1 вы уже в `/content/water-quality-ee`; при открытии другого ноутбука выполните в нём одну ячейку:

```python
%cd /content/water-quality-ee
```

или добавьте её в начало ноутбука.

**Без GitHub:** *File → Upload folder* или zip → распаковать в `/content/water-quality-ee` → `pip install` как в шаге 1 (из этой папки).